# Notebook 05 — Fine-tuning Strategies

Notebook 04 showed the simplest form of transfer learning: freeze the backbone, train a new head. That is a great baseline but it leaves accuracy on the table. This notebook teaches the techniques real practitioners use when they want to squeeze every last percentage point: **discriminative learning rates**, **LR schedulers**, the **LR range test**, **progressive unfreezing**, and **early stopping**. We finish with a full fine-tune that targets >92% val accuracy on Oxford-IIIT Pet.

## Learning goals
- Decide when to freeze vs fully fine-tune.
- Use per-parameter-group (*discriminative*) learning rates.
- Visualise and choose among StepLR, CosineAnnealingLR, OneCycleLR and warmup+cosine.
- Run a fastai-style **LR range test** to pick a sane LR.
- Implement **progressive unfreezing** one stage at a time.
- Use early stopping to save compute.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/numberonewastefellow/my_learning/blob/main/notebooks/05_finetuning_strategies.ipynb)

In [ ]:
# Universal setup: runs identically in Colab and locally
import sys, os
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("my_learning"):
        !git clone --quiet https://github.com/numberonewastefellow/my_learning.git
    %cd my_learning
    !pip install -q -r requirements.txt

repo_root = os.path.abspath(".") if IN_COLAB else os.path.abspath("..")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from utils.env import bootstrap
info = bootstrap()
device = info.device

## 1. Frozen head vs full fine-tune — when to use which?

| Situation | Recommendation |
|---|---|
| Dataset has **<1k** images / class and looks like ImageNet (natural photos, animals, objects) | **Freeze backbone**, train head only. Fast and safe. |
| Dataset has **1k–10k images** / class and mostly matches ImageNet | **Partial fine-tune**. Unfreeze the last 1–2 stages of the backbone with a small LR. |
| Dataset has **10k+ images** / class OR domain shift (medical, satellite, microscopy) | **Full fine-tune** — unfreeze everything. Use *discriminative* LRs so the pretrained lower layers don't forget. |
| Very limited GPU time | Freeze backbone. |

**Risk of full fine-tune with a big LR:** *catastrophic forgetting*. The pretrained weights get blown away within a handful of SGD steps. The standard remedies — small backbone LR, LR warmup, and progressive unfreezing — all exist precisely to avoid this.

Rest of the notebook: implement each remedy on Oxford-IIIT Pet.

In [ ]:
# Reuse exactly the same dataset/transforms as Notebook 04.
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split, Subset as TSubset
import torchvision.transforms as T
from torchvision.datasets import OxfordIIITPet
import timm

from utils.env import data_dir, checkpoints_dir

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
NUM_CLASSES = 37

train_tf = T.Compose([
    T.RandomResizedCrop(224, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_tf = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

ROOT = data_dir()
full_trainval = OxfordIIITPet(ROOT, split="trainval", target_types="category",
                              download=True, transform=train_tf)
val_base      = OxfordIIITPet(ROOT, split="trainval", target_types="category",
                              download=True, transform=val_tf)
test_ds       = OxfordIIITPet(ROOT, split="test",     target_types="category",
                              download=True, transform=val_tf)

g = torch.Generator().manual_seed(42)
n_total = len(full_trainval)
n_val = int(0.2 * n_total)
n_train = n_total - n_val
train_idx, val_idx = random_split(range(n_total), [n_train, n_val], generator=g)

class SplitDataset(torch.utils.data.Dataset):
    def __init__(self, ds, idx):
        self.ds = ds; self.idx = list(idx)
    def __len__(self): return len(self.idx)
    def __getitem__(self, i): return self.ds[self.idx[i]]

train_ds = SplitDataset(full_trainval, train_idx)
val_ds   = SplitDataset(val_base,       val_idx)
class_names = full_trainval.classes

BATCH = 32
nw = 0 if os.name == "nt" else 2
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=nw, pin_memory=info.cuda_available)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=nw, pin_memory=info.cuda_available)

print(f"train: {len(train_ds)}  val: {len(val_ds)}  classes: {len(class_names)}")

## 2. Discriminative learning rates

The backbone already has good pretrained weights — we want to **gently** nudge them. The head is random — we want to **aggressively** train it. So they deserve different learning rates.

We split the parameters into two *param groups* and give each a different LR:

- **Head LR** `= 1e-3` — standard AdamW-ish LR, head weights are random.
- **Backbone LR** `= 1e-5` — 100× smaller, preserves pretrained knowledge.

`torch.optim.AdamW` accepts a list of dicts, one per group. Any hyperparameter you leave out of a group-dict (weight_decay, betas, …) falls back to the top-level value.

In [ ]:
def build_pretrained_model(num_classes: int = NUM_CLASSES):
    m = timm.create_model("tf_efficientnetv2_s.in21k_ft_in1k", pretrained=True)
    m.reset_classifier(num_classes=num_classes)
    return m

model = build_pretrained_model().to(device)

# Split parameters into head vs backbone.
head_params = list(model.get_classifier().parameters())
head_ids = {id(p) for p in head_params}
backbone_params = [p for p in model.parameters() if id(p) not in head_ids]

optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": 1e-5},
    {"params": head_params,     "lr": 1e-3},
], weight_decay=1e-4)

for i, g in enumerate(optimizer.param_groups):
    n = sum(p.numel() for p in g["params"])
    print(f"group {i}: lr={g['lr']:.0e}  params={n/1e6:.2f} M")

## 3. Learning-rate schedulers compared visually

A scheduler changes the LR *as training progresses*. Four of the most common choices:

- **StepLR** — multiply LR by `gamma` every `step_size` epochs. Simple, a bit blunt.
- **CosineAnnealingLR** — smooth half-cosine decay from the initial LR down to `eta_min`. Popular default.
- **OneCycleLR** — warm up to a *max_lr*, then cosine-decay back down. Very effective for short training runs.
- **Linear warmup → cosine decay** — start tiny, linearly ramp up for a few epochs, then cosine-decay. Standard for transformer / large-model training.

Nothing beats *seeing* them, so let's simulate each over 20 epochs and plot together.

In [ ]:
from torch.optim.lr_scheduler import StepLR, CosineAnnealingLR, OneCycleLR, LambdaLR

def sim_schedule(make_sched, steps, base_lr):
    p = torch.nn.Parameter(torch.zeros(1))
    opt = torch.optim.SGD([p], lr=base_lr)
    sched = make_sched(opt)
    lrs = []
    for _ in range(steps):
        lrs.append(opt.param_groups[0]["lr"])
        opt.step()
        sched.step()
    return lrs

EPOCHS = 20
STEPS_PER_EPOCH = 100
TOTAL_STEPS = EPOCHS * STEPS_PER_EPOCH
BASE_LR = 1e-3

# StepLR: drop by 10x every 5 epochs  (step_size counted in "scheduler step" calls)
step_lrs = sim_schedule(
    lambda opt: StepLR(opt, step_size=5 * STEPS_PER_EPOCH, gamma=0.1),
    TOTAL_STEPS, BASE_LR)

# CosineAnnealingLR: full decay over TOTAL_STEPS
cos_lrs = sim_schedule(
    lambda opt: CosineAnnealingLR(opt, T_max=TOTAL_STEPS, eta_min=1e-6),
    TOTAL_STEPS, BASE_LR)

# OneCycleLR: classic warm-up-then-anneal
one_lrs = sim_schedule(
    lambda opt: OneCycleLR(opt, max_lr=BASE_LR, total_steps=TOTAL_STEPS,
                           pct_start=0.3, anneal_strategy="cos",
                           div_factor=25.0, final_div_factor=1e4),
    TOTAL_STEPS, BASE_LR)

# Linear warmup (first 10% of steps) then cosine decay to 0
def warmup_cosine(step, warmup=TOTAL_STEPS // 10, total=TOTAL_STEPS):
    if step < warmup:
        return step / max(1, warmup)
    pct = (step - warmup) / max(1, total - warmup)
    return 0.5 * (1.0 + np.cos(np.pi * pct))
wc_lrs = sim_schedule(
    lambda opt: LambdaLR(opt, lr_lambda=warmup_cosine),
    TOTAL_STEPS, BASE_LR)

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(TOTAL_STEPS) / STEPS_PER_EPOCH  # x-axis in epochs
ax.plot(x, step_lrs, label="StepLR (drop 10x every 5 epochs)")
ax.plot(x, cos_lrs,  label="CosineAnnealingLR")
ax.plot(x, one_lrs,  label="OneCycleLR (pct_start=0.3)")
ax.plot(x, wc_lrs,   label="Linear warmup + cosine")
ax.set_xlabel("epoch")
ax.set_ylabel("learning rate")
ax.set_title("LR schedules compared")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. LR range test (the "LR finder")

Before committing to a learning rate, we can **probe** for a good one in a single short run. The idea (Smith 2017, popularised by fastai):

1. Start with a very small LR, say `1e-7`.
2. After every mini-batch, multiply the LR by a constant `q > 1` so it grows *exponentially*.
3. Record the loss at each step.
4. Plot loss vs log-LR. You will see loss flat-line at first, then dive, then blow up.

A safe LR for full training is **10× less** than the LR where the loss was lowest — we want to be on the *downward slope*, not at the bottom (because at the bottom any noise tips us into the "blow up" regime).

We'll run the test on a **subset** of the Pet data so it finishes in seconds.

In [ ]:
def lr_range_test(model, loader, loss_fn, device,
                  start_lr=1e-7, end_lr=10.0, num_steps=100):
    """Run a fastai-style LR range test.

    Returns parallel lists of LRs and losses (one entry per minibatch).
    Stops early if loss diverges (loss > 4x min loss seen so far).
    """
    model = model.to(device).train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=start_lr)
    factor = (end_lr / start_lr) ** (1.0 / num_steps)

    lrs, losses = [], []
    best = float("inf")
    it = iter(loader)
    for step in range(num_steps):
        try:
            xb, yb = next(it)
        except StopIteration:
            it = iter(loader); xb, yb = next(it)

        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()

        cur_lr = optimizer.param_groups[0]["lr"]
        lrs.append(cur_lr); losses.append(loss.item())
        best = min(best, loss.item())
        if loss.item() > 4 * best or not np.isfinite(loss.item()):
            print(f"  diverged at step {step}, lr={cur_lr:.2e}")
            break

        for g in optimizer.param_groups:
            g["lr"] *= factor
    return lrs, losses

# Build a fresh model and loader for the test
probe_model = build_pretrained_model().to(device)

# Subset of 30 batches so the test is fast
probe_subset_idx = list(range(min(30 * BATCH, len(train_ds))))
probe_loader = DataLoader(TSubset(train_ds, probe_subset_idx),
                          batch_size=BATCH, shuffle=True, num_workers=0)

lrs, losses = lr_range_test(probe_model, probe_loader, nn.CrossEntropyLoss(),
                            device, start_lr=1e-7, end_lr=10.0, num_steps=100)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(lrs, losses)
ax.set_xscale("log")
ax.set_xlabel("learning rate (log)")
ax.set_ylabel("minibatch loss")
ax.set_title("LR range test")
ax.grid(alpha=0.3)

min_i = int(np.argmin(losses))
suggested = lrs[min_i] / 10.0
ax.axvline(lrs[min_i], color="red",   linestyle="--", label=f"min loss LR = {lrs[min_i]:.2e}")
ax.axvline(suggested,  color="green", linestyle="--", label=f"suggested LR = {suggested:.2e}")
ax.legend()
plt.tight_layout()
plt.show()
print(f"Suggested LR = {suggested:.2e}  (min-loss LR / 10)")

## 5. Progressive unfreezing

Instead of unfreezing the whole backbone at once, we unfreeze **one stage at a time**, starting from the top (the stage closest to the head). Between unfreezings we train for a short bit so the new parameters settle.

For a timm EfficientNetV2 the main feature extractor is `model.blocks`, a `ModuleList` of several stages. (Some variants also expose `model.conv_stem`, `model.bn1`, etc. — everything up to `blocks[-1]`.) Indexing `model.blocks[i]` gives us stage `i`, from low-level (`i=0`) to high-level (`i=-1`).

Rule of thumb: the higher the stage, the more task-specific its features, and the *earlier* you should unfreeze it. The stem and `blocks[0]` represent generic edges/colors — unfreeze those last, if at all.

In [ ]:
# Inspect the stage structure of the timm EfficientNetV2-S
preview = build_pretrained_model()
print(f"number of stages in .blocks = {len(preview.blocks)}")
for i, blk in enumerate(preview.blocks):
    n = sum(p.numel() for p in blk.parameters())
    print(f"  blocks[{i}]: {type(blk).__name__}  params={n/1e6:.2f} M")
del preview

In [ ]:
from utils.training import fit

def freeze_all_but_head(m):
    for p in m.parameters():
        p.requires_grad = False
    for p in m.get_classifier().parameters():
        p.requires_grad = True

def unfreeze_stage(m, stage_idx):
    """Unfreeze a single stage in m.blocks."""
    for p in m.blocks[stage_idx].parameters():
        p.requires_grad = True

def n_trainable(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

prog_model = build_pretrained_model().to(device)
freeze_all_but_head(prog_model)
print(f"phase 0 (head only)                trainable = {n_trainable(prog_model)/1e6:.2f} M")

# --- phase 0: head only, 1 epoch ---
opt0 = torch.optim.AdamW(
    [p for p in prog_model.parameters() if p.requires_grad], lr=1e-3,
)
h0 = fit(prog_model, train_loader, val_loader, nn.CrossEntropyLoss(), opt0,
         epochs=1, device=device,
         checkpoint_path=os.path.join(checkpoints_dir(), "nb05_phase0.pt"))

# --- phase 1: unfreeze last stage, 1 epoch ---
unfreeze_stage(prog_model, -1)
print(f"phase 1 (+ blocks[-1])             trainable = {n_trainable(prog_model)/1e6:.2f} M")
opt1 = torch.optim.AdamW([
    {"params": prog_model.blocks[-1].parameters(),           "lr": 1e-4},
    {"params": prog_model.get_classifier().parameters(),     "lr": 1e-3},
])
h1 = fit(prog_model, train_loader, val_loader, nn.CrossEntropyLoss(), opt1,
         epochs=1, device=device,
         checkpoint_path=os.path.join(checkpoints_dir(), "nb05_phase1.pt"))

# --- phase 2: unfreeze last two stages, 1 epoch ---
unfreeze_stage(prog_model, -2)
print(f"phase 2 (+ blocks[-2,-1])          trainable = {n_trainable(prog_model)/1e6:.2f} M")
opt2 = torch.optim.AdamW([
    {"params": prog_model.blocks[-2].parameters(),           "lr": 5e-5},
    {"params": prog_model.blocks[-1].parameters(),           "lr": 1e-4},
    {"params": prog_model.get_classifier().parameters(),     "lr": 1e-3},
])
h2 = fit(prog_model, train_loader, val_loader, nn.CrossEntropyLoss(), opt2,
         epochs=1, device=device,
         checkpoint_path=os.path.join(checkpoints_dir(), "nb05_phase2.pt"))

print()
print(f"phase 0 best val acc: {h0['best_val_acc']:.4f}")
print(f"phase 1 best val acc: {h1['best_val_acc']:.4f}")
print(f"phase 2 best val acc: {h2['best_val_acc']:.4f}")

## 6. Early stopping

Early stopping watches `val_acc` (or `val_loss`) and, if it fails to improve for `patience` consecutive epochs, halts training. Two benefits:

1. **Save compute** — no point grinding out 50 epochs if val stopped improving at epoch 8.
2. **Implicit regularisation** — stopping before overfitting locks in a better-generalising model.

`fit()` in `utils.training` already supports it via `early_stopping_patience`. To demonstrate, we ask for 15 epochs but set `patience=3`. On Pet with a frozen backbone, val typically plateaus within a few epochs and early stopping fires long before epoch 15.

In [ ]:
es_model = build_pretrained_model().to(device)
freeze_all_but_head(es_model)

opt_es = torch.optim.AdamW(
    [p for p in es_model.parameters() if p.requires_grad], lr=1e-3,
)

history_es = fit(
    es_model, train_loader, val_loader, nn.CrossEntropyLoss(), opt_es,
    epochs=15, device=device,
    early_stopping_patience=3,
    checkpoint_path=os.path.join(checkpoints_dir(), "nb05_early.pt"),
)

print("train_loss len :", len(history_es["train_loss"]), "  (<15 means ES fired)")
print("best_val_acc   :", history_es["best_val_acc"])
print("best_epoch     :", history_es["best_epoch"])

## 7. The full-fine-tune run

Putting it all together. We:

1. Load the pretrained model and swap the head.
2. Build a **discriminative-LR AdamW**: backbone 1e-5, head 1e-3.
3. Add a **CosineAnnealingLR** scheduler.
4. Train for 10 epochs with early stopping patience 3.
5. Plot curves and a confusion matrix.

Target: **>92% val accuracy** on Oxford-IIIT Pet. On a single modern GPU this run takes a few minutes; on CPU it will take much longer — consider reducing epochs if you're on CPU.

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR
from utils.plotting import plot_curves

ft_model = build_pretrained_model().to(device)

# Everything is trainable by default in timm — just build the discriminative optimizer.
head_params     = list(ft_model.get_classifier().parameters())
head_ids        = {id(p) for p in head_params}
backbone_params = [p for p in ft_model.parameters() if id(p) not in head_ids]

optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": 1e-5},
    {"params": head_params,     "lr": 1e-3},
], weight_decay=1e-4)

EPOCHS = 10
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)

history_ft = fit(
    ft_model, train_loader, val_loader, nn.CrossEntropyLoss(), optimizer,
    epochs=EPOCHS, device=device,
    scheduler=scheduler,
    early_stopping_patience=3,
    checkpoint_path=os.path.join(checkpoints_dir(), "nb05_full_ft.pt"),
)

print(f"best val acc = {history_ft['best_val_acc']:.4f} at epoch {history_ft['best_epoch']}")
plot_curves(history_ft)

In [ ]:
from collections import Counter
from sklearn.metrics import confusion_matrix
from utils.plotting import plot_confusion_matrix

ft_model.eval()
preds_all, tgts_all = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(device)
        preds_all.append(ft_model(xb).argmax(1).cpu())
        tgts_all.append(yb)
preds_all = torch.cat(preds_all).numpy()
tgts_all  = torch.cat(tgts_all).numpy()

# Top-10 most-frequent classes for a readable matrix
top10 = [c for c, _ in Counter(tgts_all.tolist()).most_common(10)]
keep = np.isin(tgts_all, top10)
t = tgts_all[keep]; p = preds_all[keep]
remap = {c: i for i, c in enumerate(top10)}
t = np.vectorize(remap.get)(t)
mask = np.isin(p, top10)
t = t[mask]; p = np.vectorize(remap.get)(p[mask])

cm = confusion_matrix(t, p, labels=list(range(10)))
plot_confusion_matrix(cm, class_names=[class_names[c] for c in top10], normalize=True)

## Key Takeaways

- **Frozen head = quick baseline. Fine-tuning = extra accuracy.** Choose based on data size and domain shift.
- **Discriminative LRs** (small for backbone, large for head) are essential for fine-tuning without catastrophic forgetting.
- **Schedulers matter.** Cosine and OneCycle usually beat constant LR. Linear warmup is crucial for very high LRs or large models.
- **LR range test** gives you a principled way to pick an LR in <1 minute.
- **Progressive unfreezing** is a gentler alternative to unfreezing everything at once — useful on very small or domain-shifted datasets.
- **Early stopping** saves compute and reduces overfitting; always set a reasonable `patience`.
- With all the above combined, EfficientNetV2-S reliably hits **>92%** on Oxford-IIIT Pet in ~10 epochs.

## Exercises

1. **Compare schedulers.** Run full fine-tune with `OneCycleLR` vs `CosineAnnealingLR`. Plot both val-acc curves on the same axis.
2. **Tune discriminative LRs.** Try backbone LR in `{1e-6, 1e-5, 5e-5}` and head LR in `{5e-4, 1e-3, 3e-3}`. Which combo is best?
3. **LR finder v2.** Modify `lr_range_test` to track a *smoothed* loss (EMA with β=0.98) and use the steepest-descent point rather than min-loss. Compare to the original.
4. **Progressive unfreezing, all the way down.** Extend to 5 phases so every stage gets unfrozen. Does val accuracy keep climbing?
5. **Test-set eval.** Load the best checkpoint, evaluate on the `test` split, and report top-1 + top-5 accuracy using `utils.metrics.MetricTracker`.
6. **Label smoothing.** Replace `CrossEntropyLoss()` with `CrossEntropyLoss(label_smoothing=0.1)`. Measure the effect on val accuracy + calibration.